In [0]:
from pyspark.sql import functions as F

# 1. Configuração de  caminhos
path_bronze = "dbfs:/Volumes/workspace/default/data/delta/bronze/"
path_silver = "dbfs:/Volumes/workspace/default/data/delta/silver/"

try:
    # 2. Carregar tabelas (Certifique-se que os nomes batem com a sua Bronze)
    orders = spark.read.format("delta").load(f"{path_bronze}orders")
    order_items = spark.read.format("delta").load(f"{path_bronze}order_items")
    customers = spark.read.format("delta").load(f"{path_bronze}customers")
    products = spark.read.format("delta").load(f"{path_bronze}products")
    sellers = spark.read.format("delta").load(f"{path_bronze}sellers")

    # 3. Regra de Negócio: Filtro de Status
    orders_filtered = orders.filter(F.col("order_status").isin("delivered", "shipped"))

    # 4. Tratamento de Colunas Duplicadas (Renomeando Timestamps para evitar erro de Metadata)
    # Fazemos isso antes do Join
    order_items = order_items.withColumnRenamed("ingestion_timestamp", "ingestion_timestamp_items")
    customers = customers.withColumnRenamed("ingestion_timestamp", "ingestion_timestamp_cust")
    products = products.withColumnRenamed("ingestion_timestamp", "ingestion_timestamp_prod")
    sellers = sellers.withColumnRenamed("ingestion_timestamp", "ingestion_timestamp_sell")

    # 5. Join de Consolidação
    silver_consolidated = orders_filtered \
        .join(order_items, "order_id", "inner") \
        .join(customers, "customer_id", "inner") \
        .join(products, "product_id", "left") \
        .join(sellers, "seller_id", "left")

    # 6. Salvamento com overwriteSchema para evitar o erro DELTA_METADATA_MISMATCH
    silver_consolidated.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(f"{path_silver}orders_consolidated")

    print("✅ Silver: Tabela 'orders_consolidated' gerada com sucesso.")

except Exception as e:
    print(f"❌ Erro detectado no processamento da Silver: {str(e)}")
    raise e # Re-lança o erro para o orquestrador registrar a falha

✅ Silver: Tabela 'orders_consolidated' gerada com sucesso.
